[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Regression Metrics &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (max_error, mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, median_absolute_error, r2_score, root_mean_squared_error)
from sklearn.model_selection import KFold, cross_val_predict, train_test_split

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


In [2]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

linear = LinearRegression().fit(X_train, y_train)
forest = RandomForestRegressor(random_state=0).fit(X_train, y_train)

The split and the two models are those of the worked examples, since this notebook runs on its own.

**1.** Print the forest's MAE and RMSE on the held-back stations, rounded to three places, and the difference between them.

In [3]:
predicted = forest.predict(X_test)
mae = mean_absolute_error(y_test, predicted)
rmse = root_mean_squared_error(y_test, predicted)

print(round(mae, 3), round(rmse, 3), round(rmse - mae, 3))

0.752 0.972 0.221


RMSE is larger, because squaring makes the forest's largest misses count for more.

**2.** Print the linear model's R² and MAE on the held-back stations north of latitude 68, and how many of them there are.

In [4]:
north = X_test["latitude"] > 68
predicted = linear.predict(X_test[north])

print(round(r2_score(y_test[north], predicted), 3), round(mean_absolute_error(y_test[north], predicted), 3))
print(int(north.sum()), "stations")
print("all held-back stations:", round(r2_score(y_test, linear.predict(X_test)), 3),
      round(mean_absolute_error(y_test, linear.predict(X_test)), 3))
print("spread of their January means:", round(y_test[north].std(), 2), "| of all:", round(y_test.std(), 2))

0.829 0.789
63 stations
all held-back stations: 0.888 0.784
spread of their January means: 2.41 | of all: 3.0


The northern stations' MAE is about the same as all the stations', and their R² is lower, because January varies less among them.

**3.** Print the linear model's mean residual on the held-back stations for each `owner`, rounded to two places.

In [5]:
residuals = y_test - linear.predict(X_test)
owner = network.loc[y_test.index, "owner"]

print(residuals.groupby(owner).mean().round(2))

owner
airport    0.14
met       -0.15
private    0.17
road      -0.32
Name: jan_mean, dtype: float64


They range from -0.32 to 0.17, much closer together than the terrain biases of the worked example, which ran from -0.7 to 0.88. The owner had no part in making the January means, so no owner was expected to stand out.

**4.** Fit `DummyRegressor(strategy="median")` on the training stations and print its MAE on the held-back stations beside the mean-only model's.

In [6]:
median_only = DummyRegressor(strategy="median").fit(X_train, y_train)
mean_only = DummyRegressor().fit(X_train, y_train)

print(round(mean_absolute_error(y_test, median_only.predict(X_test)), 3),
      round(mean_absolute_error(y_test, mean_only.predict(X_test)), 3))
print("on the training stations:", round(mean_absolute_error(y_train, median_only.predict(X_train)), 4),
      round(mean_absolute_error(y_train, mean_only.predict(X_train)), 4))

2.333 2.33
on the training stations: 2.3667 2.3668


On the training stations, the median is the single value with the smallest average absolute distance to them, so on the training stations it can only tie or beat the mean, and the last line shows it doing so. On the held-back stations the two come out at 2.333 and 2.33, as good as each other.

**5.** Show the five held-back stations the forest misses most, with their terrain, measured value and residual.

In [7]:
forest_residuals = y_test - forest.predict(X_test)
worst = forest_residuals.abs().nlargest(5).index

network.loc[worst, ["station", "terrain", "jan_mean"]].assign(residual=forest_residuals[worst].round(2))

,station,terrain,jan_mean,residual
533,ST0534,inland,1.7,3.24
867,ST0868,inland,-5.1,3.11
975,ST0976,mountain,-13.0,-2.66
1053,ST1054,inland,-8.7,-2.58
745,ST0746,fjord,-8.6,-2.51


`nlargest` keeps the first of any tied values in the table's order, so the five are the same on every run.

---

&#8592; **Back to:** [Regression Metrics](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/07-regression-metrics.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
